In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")


In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("diabetes.pdf")

docs = loader.load()

C:\Users\jaysi\AppData\Local\Temp\ipykernel_3676\3114154388.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
len(docs)

23

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)

text = text_splitter.split_documents(docs)
text

len(text)

179

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

d:\Code\06.Generative AI\04 Medical Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2016.74it/s]


In [6]:
persist_directory = "medical_db"

from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="medical_db"
)

print("Documents stored successfully")

Documents stored successfully


In [7]:
retriever = vectordb.as_retriever(
    search_kwargs={"k": 3}
)

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key,
    temperature=0
    )

In [9]:
query = "what is diabetis"
retriever.invoke(query)

[Document(id='6ad05ceb-a8a2-492c-98e5-ac9c206aaf7a', metadata={'source': 'diabetes.pdf', 'author': 'Ramchandra', 'title': '1. Diabetes-24-04-19.cdr', 'moddate': '2023-08-07T15:29:16+05:30', 'creationdate': '2019-04-26T11:58:48+05:30', 'page': 0, 'creator': 'CorelDRAW X5', 'total_pages': 23, 'page_label': '1', 'producer': 'Corel PDF Engine Version 15.0.0.486'}, page_content='Diabetes'),
 Document(id='e56a614f-33a3-4539-93b4-8d9e61d3fa47', metadata={'moddate': '2023-08-07T15:29:16+05:30', 'title': '1. Diabetes-24-04-19.cdr', 'source': 'diabetes.pdf', 'page': 0, 'producer': 'Corel PDF Engine Version 15.0.0.486', 'creationdate': '2019-04-26T11:58:48+05:30', 'total_pages': 23, 'page_label': '1', 'author': 'Ramchandra', 'creator': 'CorelDRAW X5'}, page_content='Diabetes'),
 Document(id='2ab0e054-c8ee-413e-ae45-79544d5b412d', metadata={'producer': 'Corel PDF Engine Version 15.0.0.486', 'source': 'diabetes.pdf', 'page_label': '4', 'moddate': '2023-08-07T15:29:16+05:30', 'creator': 'CorelDRAW X

In [10]:
retriever.search_type

'similarity'

In [11]:
template="""
<|context|>
You are an Medical Assistant.Please be truthful and give answers.
</s>
<|user|>
{query}
</s>
<|assistant|>
"""

In [12]:
from langchain_core.runnables import RunnablePassthrough

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

from langchain_classic.chains import RetrievalQA, LLMChain

In [15]:
prompt = ChatPromptTemplate.from_template(template)

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

In [16]:
rag_chain = (
    {"context":retriever, "query":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
    )

In [17]:
result = rag_chain.invoke(query)
print(result)


**Diabetes mellitus** (often simply called “diabetes”) is a chronic metabolic disorder in which the body has difficulty regulating blood‑glucose (sugar) levels. This can happen because:

1. **The pancreas does not produce enough insulin** – the hormone that helps cells take up glucose from the bloodstream (insulin deficiency), **or**  
2. **The body’s cells become resistant to insulin’s effects** (insulin resistance), so glucose cannot enter the cells efficiently.

When glucose stays elevated in the blood (hyperglycemia), it can damage blood vessels, nerves, and organs over time.

---

### Main Types of Diabetes

| Type | Key Features | Typical Onset | Typical Treatment |
|------|--------------|---------------|-------------------|
| **Type 1** | Autoimmune destruction of insulin‑producing β‑cells → little or no insulin. | Usually childhood, adolescence, or early adulthood, but can appear at any age. | Lifelong insulin therapy (multiple daily injections or pump), blood‑glucose monitorin

In [19]:
while True:
  query = input("Input query: ")
  if query == "exit":
    break
  result= rag_chain.invoke(query)
  print("Answer:",result)

Answer: **Diabetes mellitus** (often simply called “diabetes”) is a chronic medical condition in which the body has difficulty regulating blood glucose (sugar) levels. Glucose is the main source of energy for the body’s cells, and its level in the bloodstream is normally kept within a narrow range by the hormone insulin, which is produced by the pancreas.

### How Diabetes Develops
- **Insufficient insulin production** – The pancreas does not make enough insulin.
- **Insulin resistance** – The body’s cells do not respond properly to the insulin that is produced.
- **Both** – In many cases, especially in type 2 diabetes, both reduced insulin secretion and insulin resistance are present.

When glucose cannot enter cells effectively, it builds up in the blood (hyperglycemia). Over time, high blood glucose can damage blood vessels, nerves, and organs.

### Main Types of Diabetes
| Type | Key Features | Typical Onset | Common Risk Factors |
|------|--------------|---------------|-----------